In [1]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set_theme(style="whitegrid")

pd.set_option('display.max_columns', None)

pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================

train_path = "../../data/raw/train.csv"
test_path = "../../data/raw/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

In [3]:
# ============================================================
# 3. 결측 현황 확인 함수
# ============================================================
# 결측치 개수와 결측 비율을 함께 보기 위한 함수를 만든다.

def missing_table(df):
    missing_count = df.isnull().sum()
    missing_ratio = (df.isnull().sum() / len(df)) * 100

    result = pd.DataFrame({
        'missing_count' : missing_count,
        'missing_ratio' : missing_ratio
    })

    # 결측이 있는 컬럼만 남기고 결측 개수 기준으로 내림차순 정렬한다.
    result = result[result['missing_count'] > 0].sort_values(by = 'missing_count', ascending = False)

    return result

In [4]:
# ============================================================
# 4. 결측 처리 전 상태 확인
# ============================================================

print("[train 결측 현황]")
display(missing_table(train))

print("\n[test 결측 현황]")
display(missing_table(test))

[train 결측 현황]


,missing_count,missing_ratio
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22



[test 결측 현황]


,missing_count,missing_ratio
Cabin,327,78.23
Age,86,20.57
Fare,1,0.24


In [8]:
# ============================================================
# 5. 결측 처리 기준 확인
# ============================================================

# Age는 Pclass와 Sex에 따라 차이가 있는지 확인한다.
print("[train 기준 Pclass + Sex별 Age 중앙값]")
display(train.groupby(['Pclass','Sex'])['Age'].median().unstack())

# Embarked의 최빈값을 확인한다.
embarked_mode = train['Embarked'].mode()[0]
print("\n[train Embarked 최빈값]")
print(embarked_mode)

# Fare의 train 기준 중앙값을 확인한다.
fare_median = train['Fare'].median()
print("\n[train Fare 중앙값]")
print(fare_median)

# Cabin의 결측 비율이 높으므로 원본 상태를 간단히 확인한다.
print("\n[train Cabin 예시]")
print(train['Cabin'].head())

[train 기준 Pclass + Sex별 Age 중앙값]


Sex,female,male
Pclass,,
1,35.00,40.00
2,28.00,30.00
3,21.50,25.00



[train Embarked 최빈값]
S

[train Fare 중앙값]
14.4542

[train Cabin 예시]
0     NaN
1     C85
2     NaN
3    C123
4     NaN
Name: Cabin, dtype: object


In [17]:
# ============================================================
# 6. 복사본 생성
# ============================================================

train_mv = train.copy()
test_mv = test.copy()

In [18]:
# ============================================================
# 7. Age 결측 처리 함수
# ============================================================

# train 기준으로 만든 그룹별 중앙값을 이용해 Age 결측을 채우는 함수를 만든다.
def fill_age_by_train_map(df, age_median_map, fallback_median):
    df = df.copy()

    for(pclass, sex), median_age in age_median_map.items():
        df.loc[
            (df['Age'].isnull()) &
            (df['Pclass'] == pclass) &
            (df['Sex'] == sex),
            'Age'
            ] = median_age

        # 혹시 그룹 기준으로도 안 채워진 값은 train 전체 중앙값으로 채운다. 
        df['Age'] = df['Age'].fillna(fallback_median)

        return df

# train 기준 Age 중앙값 맵을 만든다.
age_median_map = train.groupby(['Pclass','Sex'])['Age'].median()

# train 전체 Age 중앙값을 fallback 값으로 사용한다.
age_global_median = train['Age'].median()

In [24]:
# ============================================================
# 8. 결측 처리 수행
# ============================================================

# Age는 train 기준 그룹 중앙값으로 채운다.
train_mv = fill_age_by_train_map(train_mv, age_median_map, age_global_median)
test_mv = fill_age_by_train_map(test_mv, age_median_map, age_global_median)

# Embarked는 train 최빈값으로 채운다.
train_mv['Embarked'] = train_mv['Embarked'].fillna(embarked_mode)
test_mv['Embarked'] = test_mv['Embarked'].fillna(embarked_mode)

# Fare는 train 중앙값으로 채운다.
train_mv['Fare'] = train_mv['Fare'].fillna(fare_median)
test_mv['Fare'] = test_mv['Fare'].fillna(fare_median)

# Cabin은 결측 자체를 정보로 보기 위해 Missing으로 채운다.
train_mv['Cabin'] = train_mv['Cabin'].fillna('Missing')
test_mv['Cabin'] = test_mv['Cabin'].fillna('Missing')

In [25]:
# ============================================================
# 9. 결측 처리 후 상태 확인
# ============================================================

print("[결측 처리 후 train 결측 현황]")
display(missing_table(train_mv))

print("\n[결측 처리 후 test 결측 현황]")
display(missing_table(test_mv))

[결측 처리 후 train 결측 현황]


,missing_count,missing_ratio



[결측 처리 후 test 결측 현황]


,missing_count,missing_ratio


In [27]:
# ============================================================
# 10. 처리 결과 샘플 확인
# ============================================================

print("[train 처리 결과 샘플]")
display(train_mv.head())

print("[test 처리 결과 샘플]")
display(test_mv.head())

[train 처리 결과 샘플]


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.00,1,0,A/5 21171,7.25,Missing,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00,1,0,PC 17599,71.28,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.00,0,0,STON/O2. 3101282,7.92,Missing,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00,1,0,113803,53.10,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.00,0,0,373450,8.05,Missing,S


[test 처리 결과 샘플]


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.50,0,0,330911,7.83,Missing,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.00,1,0,363272,7.00,Missing,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.00,0,0,240276,9.69,Missing,Q
3,895,3,"Wirz, Mr. Albert",male,27.00,0,0,315154,8.66,Missing,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.00,1,1,3101298,12.29,Missing,S


In [29]:
# ============================================================
# 11. 결과 저장
# ============================================================

train_mv.to_csv("../../data/interim/train_missing_handled.csv", index=False)
test_mv.to_csv("../../data/interim/test_missing_handled.csv", index=False)

print("train_missing_handled.csv 저장 완료")
print("test_missing_handled.csv 저장 완료")

train_missing_handled.csv 저장 완료
test_missing_handled.csv 저장 완료
